# 00 · Start here

These notebooks rebuild the study's models from the EDHS 2024-25 Individual Recode: who is in
the cohort, how the three outcomes and the 21 predictors were made, how the women were split,
how each model was tuned and checked, what each model leans on, and the planner's levers. Run
them in order; each reads only what the earlier ones wrote and ends by checking its results
against the study's (in `reference/`).

The study predicts whether a woman reaches **one or more** (ANC1+), **four or more** (ANC4+)
and **eight or more** (ANC8+) antenatal care contacts, from what is known before her care
starts. Three models, **logistic regression, random forest and XGBoost**, are fitted on the
**21 predictors** at each of the three rungs: nine models. Each is tuned over the small grid
of settings the study fixed in advance.

## Before you start

1. **The data.** Obtain `ETIR8AFL.DTA` (Ethiopia DHS 2024-25, Individual Recode, Stata) from
   The DHS Program and put it in `data/`. It is licensed to you and must not be shared;
   nothing in this folder contains survey rows, and everything the notebooks write stays in
   `outputs/`, `results/` and `quick/`, which are never committed.
2. **The environment.** Python 3.12 or 3.13, then `pip install -r requirements.txt`.
3. **The order.** The table below; about half an hour in all on an 8-core laptop.

| Notebook | Does | Time |
|---|---|---|
| 01 | the cohort, the outcome ladder and the 21 predictors, from the survey file | 2 min |
| 02 | the analysis frame: the three rungs, birth timing, the women who started care | 2 min |
| 03 | the held-out split and the development folds | under 1 min |
| 04 | logistic regression, tuned at each rung | under 1 min |
| 05 | random forest, tuned at each rung | 10 min |
| 06 | XGBoost, tuned at each rung | 3 min |
| 07 | evaluation on the held-out women | 10 min |
| 08 | explanations: what each model leans on | 2 min |
| 09 | the planner's levers and why regions differ | 5 min |

To run them all without opening Jupyter, from this folder:

```
python scripts/run_notebooks.py            # all of them, in order
python scripts/run_notebooks.py --from 05  # resume after a stop
python scripts/run_notebooks.py --quick    # a fast check with fewer resamples
```

Executed copies, with every table, figure and check, land in `executed/`.

## What else the study tried

The study also screened every column of the survey into a candidate pool, compared four
predictor sets, fitted eleven model families and two neural networks, recalibrated and combined
them. None of that is needed for these models; `reproduction_full/` rebuilds all of it, and
`research_record/` records its results.

## Folders

```
data/            put ETIR8AFL.DTA here
notebooks/       00-09
src/anc8/        the analysis package every notebook uses
scripts/         the stage scripts some notebooks run
reference/       the study's results, for the checks
outputs/         what the notebooks write (analysis frames, tables, figures)
results/         the nine models' predictions and settings
```

In [ ]:
# Is the environment the one the study used, and is the survey file in place?
import importlib, sys
from pathlib import Path

here = Path.cwd()
project = here.parent if here.name.startswith("notebooks") else here
want = {}
for line in (project / "requirements.txt").read_text().splitlines():
    if "==" in line and not line.startswith("#"):
        name, v = line.split("==")
        want[name.strip()] = v.strip()

MODULE = {"scikit-learn": "sklearn", "python-docx": "docx"}
print(f"python {sys.version.split()[0]}")
for name, v in want.items():
    try:
        m = importlib.import_module(MODULE.get(name, name))
        got = getattr(m, "__version__", "?")
        print(f"  {name:14s} {got:10s} {'' if got == v else f'(the study used {v})'}")
    except ImportError:
        print(f"  {name:14s} MISSING    (pip install -r requirements.txt)")

dta = [p for p in (project / "data").glob("*") if p.name.lower() == "etir8afl.dta"]
print("\nsurvey file:", dta[0] if dta else "NOT FOUND: put ETIR8AFL.DTA in data/")